# Group60 NB5 — Merge Results + Error Analysis + Final Table
**Kaggle. No GPU needed. ~10 minutes.**

**Run this AFTER NB1-4 are complete.**

**How to use:**
1. Upload this notebook to Kaggle
2. Add each completed notebook's output as a Dataset:
   - In each previous notebook: Output panel → `+` → Add as Dataset → name it e.g. `group60-nb1`
   - In THIS notebook: Add Data (right panel) → Your Datasets → add all four
3. Settings → Accelerator → None → Run All

The notebook will find all result JSON files automatically.

### Step 1 — Install

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'langdetect', 'lime', 'accelerate'], check=True)
print("Packages ready.")

### Step 2 — Setup

In [ ]:
import os, random, warnings, json, shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.pipeline import Pipeline

import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    EarlyStoppingCallback, DataCollatorWithPadding,
    set_seed as hf_set_seed
)
from datasets import Dataset as HFDataset
from langdetect import detect, LangDetectException

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
hf_set_seed(SEED)

LANGUAGES     = {"hau":"Hausa","yor":"Yoruba","ibo":"Igbo","pcm":"Nigerian Pidgin","swa":"Swahili"}
LABEL_MAP     = {"positive":2,"neutral":1,"negative":0}
LABEL_NAMES   = ["negative","neutral","positive"]
CMI_THRESHOLD = 0.20
MODEL_NAME    = "Davlan/afro-xlmr-base"
MBERT_NAME    = "google-bert/bert-base-multilingual-cased"
RUN_ID        = datetime.now().strftime("%Y%m%d_%H%M")

# Kaggle: /kaggle/working is the persistent output directory
# Everything saved here is downloadable from the Output panel on the right
OUTPUT_DIR = Path("/kaggle/working/group60_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN   = 64
BATCH_SIZE= 32
GRAD_ACCUM= 1
LR        = 2e-5
N_EPOCHS  = 5
PATIENCE  = 2
RESULTS   = {code:{} for code in LANGUAGES}

def save_checkpoint(label="ckpt"):
    """Print what has been saved so far - Kaggle auto-saves /kaggle/working."""
    files = list(OUTPUT_DIR.glob("*"))
    print(f"  [{label}] {len(files)} files in output dir. Kaggle auto-saves these.")
    for f in sorted(files)[-5:]:
        print(f"    {f.name}")

def evaluate_on_subsets(predict_fn, code, model_name):
    RESULTS[code][model_name] = {}
    for sname, df in [("full",DATA[code]["test"]),
                       ("mono",DATA[code]["test_mono"]),
                       ("mixed",DATA[code]["test_mixed"])]:
        if len(df)==0: RESULTS[code][model_name][sname]=None; continue
        yt = df["label_int"].values
        yp = predict_fn(df["tweet"].tolist())
        RESULTS[code][model_name][sname] = {
            "weighted_f1": round(f1_score(yt,yp,average="weighted",zero_division=0)*100,2),
            "macro_f1":    round(f1_score(yt,yp,average="macro",   zero_division=0)*100,2),
            "accuracy":    round(accuracy_score(yt,yp)*100,2),
            "n_samples":   len(df),
            "report":      classification_report(yt,yp,target_names=LABEL_NAMES,
                                                  zero_division=0,output_dict=True),
        }
        r=RESULTS[code][model_name][sname]
        print(f"    [{sname:5s}] n={len(df):,}  wF1={r['weighted_f1']}  mF1={r['macro_f1']}")

def compute_metrics(ep):
    logits,labels = ep
    if isinstance(logits,tuple): logits=logits[0]
    preds = np.argmax(logits,axis=-1)
    return {"weighted_f1":f1_score(labels,preds,average="weighted",zero_division=0),
            "macro_f1":   f1_score(labels,preds,average="macro",   zero_division=0),
            "accuracy":   accuracy_score(labels,preds)}

def df_to_hf(df, tokenizer):
    d = HFDataset.from_pandas(df[["tweet","label_int"]].rename(columns={"label_int":"labels"}))
    def tok_fn(ex, t=tokenizer):
        return t(ex["tweet"], truncation=True, max_length=MAX_LEN, padding=False)
    d = d.map(tok_fn, batched=True, remove_columns=["tweet"])
    d.set_format("torch"); return d

def get_training_args(output_dir):
    return TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=N_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LR, weight_decay=0.01, warmup_steps=200,
        gradient_accumulation_steps=GRAD_ACCUM, lr_scheduler_type="linear",
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="weighted_f1",
        greater_is_better=True, save_total_limit=1, seed=SEED,
        logging_steps=50, report_to="none",
        fp16=torch.cuda.is_available(), dataloader_num_workers=2,
    )

def run_inference(model, tokenizer, texts, batch=64):
    preds=[]; model.eval()
    for i in range(0, len(texts), batch):
        enc = tokenizer(texts[i:i+batch], truncation=True, max_length=MAX_LEN,
                        padding=True, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            preds.extend(model(**enc).logits.argmax(-1).cpu().tolist())
    return preds

def save_results_json(tag=""):
    def clean(o):
        if isinstance(o,dict): return {k:clean(v) for k,v in o.items()}
        if isinstance(o,(int,float,str,bool,type(None))): return o
        if isinstance(o,np.integer): return int(o)
        if isinstance(o,np.floating): return float(o)
        if isinstance(o,np.ndarray): return o.tolist()
        return str(o)
    path = OUTPUT_DIR / f"Group60_Results{tag}_{RUN_ID}.json"
    with open(path,"w") as f: json.dump(clean(RESULTS),f,indent=2)
    print(f"  Saved: {path.name}")
    return path

print(f"Device   : {DEVICE}")
print(f"Run ID   : {RUN_ID}")
print(f"Output   : {OUTPUT_DIR}")
print("Setup complete.")


### Step 3 — Load data + CMI (for error analysis)

In [ ]:
GITHUB_BASE = "https://raw.githubusercontent.com/afrisenti-semeval/afrisent-semeval-2023/main/data"
SPLIT_FILES = {"train":"train.tsv","validation":"dev.tsv","test":"test.tsv"}
DATA = {}

def load_tsv(code, split):
    return pd.read_csv(f"{GITHUB_BASE}/{code}/{SPLIT_FILES[split]}",
                       sep="\t", header=0, on_bad_lines="skip")

def normalise(df):
    df=df.copy(); col_map={}
    for c in df.columns:
        if c.lower() in ("text","tweet"):        col_map[c]="tweet"
        elif c.lower() in ("label","sentiment"): col_map[c]="label"
    df=df.rename(columns=col_map)
    df["tweet"]=df["tweet"].astype(str).str.strip()
    df["label"]=df["label"].astype(str).str.strip().str.lower()
    df=df[df["label"].isin(LABEL_MAP)].copy()
    df["label_int"]=df["label"].map(LABEL_MAP)
    return df.reset_index(drop=True)

for code,name in LANGUAGES.items():
    print(f"Loading {name}...", end=" ", flush=True)
    DATA[code]={s:normalise(load_tsv(code,s)) for s in ["train","validation","test"]}
    tr,va,te=DATA[code]["train"],DATA[code]["validation"],DATA[code]["test"]
    print(f"train={len(tr):,}  val={len(va):,}  test={len(te):,}")
print("Data loaded.")

def is_valid_token(tok):
    tok=tok.strip(".,!?;:\"'()[]")
    return tok and len(tok)>=3 and not tok.startswith(("@","http")) and not tok.isdigit()

def _detect_en(tok):
    try: return detect(tok)=="en"
    except LangDetectException: return False

def english_token_ratio(tweet):
    valid=[t for t in tweet.split() if is_valid_token(t)]
    if not valid: return 0.0
    return sum(1 for t in valid if _detect_en(t)) / len(valid)

for code,name in LANGUAGES.items():
    print(f"  CMI {name}...", end=" ", flush=True)
    df=DATA[code]["test"].copy()
    df["en_ratio"]=df["tweet"].apply(english_token_ratio)
    df["code_mixed"]=df["en_ratio"]>CMI_THRESHOLD
    DATA[code]["test"]=df
    DATA[code]["test_mono"]=df[~df["code_mixed"]].reset_index(drop=True)
    DATA[code]["test_mixed"]=df[df["code_mixed"]].reset_index(drop=True)
    n,nm=len(df),df["code_mixed"].sum()
    print(f"mono={n-nm}  mixed={nm} ({100*nm/n:.1f}%)")
print("CMI done.")


### Step 4 — Find and merge all result JSONs from previous notebooks

In [ ]:
import glob

# Kaggle mounts added datasets under /kaggle/input/
# This finds every Group60_Results_*.json across all added datasets
print("Searching for result JSONs...")
search_paths = [
    "/kaggle/input/**/*.json",
    "/kaggle/working/**/*.json",
]
all_json_files = []
for pattern in search_paths:
    all_json_files.extend(glob.glob(pattern, recursive=True))

result_files = [f for f in all_json_files if "Group60_Results" in f]
print(f"Found {len(result_files)} result files:")
for f in result_files:
    print(f"  {f}")

# Merge all into RESULTS
RESULTS = {code:{} for code in LANGUAGES}
for jf in sorted(result_files):
    with open(jf) as f:
        partial = json.load(f)
    for code in LANGUAGES:
        if code in partial:
            for model_name, model_data in partial[code].items():
                if model_name not in RESULTS[code]:
                    RESULTS[code][model_name] = model_data
                    print(f"  Loaded {model_name} for {LANGUAGES[code]}")

print("\nAvailable results per language:")
for code, name in LANGUAGES.items():
    print(f"  {name}: {list(RESULTS[code].keys())}")


### Step 5 — Final Results Table

In [ ]:
rows=[]
for code,name in LANGUAGES.items():
    for model in ["tfidf_lr","mbert","laft","maft"]:
        if model not in RESULTS[code]: continue
        for subset in ["full","mono","mixed"]:
            r=RESULTS[code][model].get(subset)
            if not r: continue
            rows.append({
                "Language": name,
                "Model":    model.upper().replace("_","+"),
                "Subset":   subset,
                "n":        r["n_samples"],
                "wF1":      r["weighted_f1"],
                "mF1":      r["macro_f1"],
                "Acc":      r["accuracy"],
            })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))
out_path = OUTPUT_DIR / f"Group60_Results_FINAL_{RUN_ID}.csv"
results_df.to_csv(out_path, index=False)
print(f"\nSaved: {out_path.name}")


### Step 6 — Error Analysis (LIME)

In [ ]:
try:
    from lime.lime_text import LimeTextExplainer
    LIME_OK = True
    print("LIME available.")
except ImportError:
    LIME_OK = False

error_rows = []
for code, name in LANGUAGES.items():
    print(f"\n{name} ({code})")
    # Find the TF-IDF pipeline across all added datasets
    pkl_files = glob.glob(f"/kaggle/input/**/Group60_TFIDF_{code}_*.pkl", recursive=True)
    pkl_files += glob.glob(f"/kaggle/working/**/Group60_TFIDF_{code}_*.pkl", recursive=True)
    if not pkl_files:
        print(f"  No TF-IDF pkl found for {name}, skipping error analysis.")
        continue
    pkl_files.sort(); pkl = pkl_files[-1]
    print(f"  Loading: {pkl.split('/')[-1]}")
    pipe = joblib.load(pkl)

    test_df = DATA[code]["test"].copy()
    y_pred  = pipe.predict(test_df["tweet"].tolist())
    inv     = {v:k for k,v in LABEL_MAP.items()}
    test_df["pred_int"]   = y_pred
    test_df["pred_label"] = pd.Series(y_pred).map(inv).values
    miss = test_df[test_df["label_int"]!=test_df["pred_int"]].copy().reset_index(drop=True)
    print(f"  Misclassified: {len(miss)}/{len(test_df)} ({100*len(miss)/len(test_df):.1f}%)")

    def auto_tag(tweet, er):
        if er>CMI_THRESHOLD: return "code_switching"
        if "?" in tweet and len(tweet.split())<8: return "sarcasm"
        if any(ch*3 in tweet.lower() for ch in "abcdefghijklmnopqrstuvwxyz"): return "orthographic_variation"
        return "other"

    for _,row in miss.sample(min(30,len(miss)),random_state=SEED).iterrows():
        er=row.get("en_ratio",0.0)
        error_rows.append({"language":name,"tweet":row["tweet"][:100],
                            "true":row["label"],"pred":row["pred_label"],
                            "en_ratio":round(er,3),"error_type":auto_tag(row["tweet"],er)})

    if LIME_OK and len(miss)>0:
        explainer = LimeTextExplainer(class_names=LABEL_NAMES, random_state=SEED)
        lime_out=[]
        for _,row in miss.head(5).iterrows():
            try:
                exp=explainer.explain_instance(row["tweet"],pipe.predict_proba,
                                                num_features=6,num_samples=300)
                lime_out.append({"tweet":row["tweet"][:80],"true":row["label"],
                                  "pred":row["pred_label"],"features":exp.as_list()})
            except Exception: pass
        with open(OUTPUT_DIR/f"Group60_LIME_{code}_{RUN_ID}.json","w") as f:
            json.dump(lime_out,f,indent=2,ensure_ascii=False)
        print(f"  LIME saved.")

err_df = pd.DataFrame(error_rows)
err_df.to_csv(OUTPUT_DIR/f"Group60_ErrorAnalysis_{RUN_ID}.csv",index=False)
if len(err_df):
    print("\nError type distribution:")
    print(err_df.groupby(["language","error_type"]).size().unstack(fill_value=0))
print("\nError analysis done.")


### Step 7 — Zip everything for download

In [ ]:
zip_path = f"/kaggle/working/Group60_AfriSenti_FINAL_{RUN_ID}"
shutil.make_archive(zip_path, "zip", str(OUTPUT_DIR))
print(f"Created: {zip_path}.zip")
print("\nDownload from the Output panel on the right side of the screen.")
print(f"Run ID: {RUN_ID}")
print("\nKey files in the zip:")
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"  {f.name}")
for f in sorted(OUTPUT_DIR.glob("*.json")):
    print(f"  {f.name}")
